# 1. Environment Setup

## 1.1 Clone Repository & Install Dependencies

In [1]:
!git clone https://github.com/11erlangga/legal-rag-slm.git /kaggle/working/repo

Cloning into '/kaggle/working/repo'...
remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 10 (delta 1), reused 10 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (10/10), done.
Resolving deltas: 100% (1/1), done.


In [2]:
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 10.8 MB/s eta 0:00:00
  Attempting uninstall: dill
    Found existing installation: dill 0.4.1
    Uninstalling dill-0.4.1:
      Successfully uninstalled dill-0.4.1
  Attempting uninstall: datasets
    Found existing installation: datasets 5.0.0
    Uninstalling datasets-5.0.0:
      Successfully uninstalled datasets-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.8/110.8 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 MB 35.

# 1.2 Import Libraries & Secrets

In [3]:
import sys
sys.path.append('/kaggle/working/repo/src')

from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
import torch
import wandb
from transformers import TextStreamer

from model_utils import load_model_and_tokenizer, apply_lora
from data_utils import prepare_datasets

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
WANDB_TOKEN = user_secrets.get_secret("WANDB_TOKEN")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
if torch.cuda.is_available():
    gpu_stats = torch.cuda.get_device_properties(0)
    max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
    print(f"GPU Detected: {gpu_stats.name}. Max Memory = {max_memory} GB.")
else:
    print("WARNING: GPU Not Detected.")

GPU Detected: Tesla T4. Max Memory = 14.562 GB.


# 2. Experiment Tracking (WandB)

In [5]:
EXPERIMENT_NAME = "sft-qwen25-3b-run2"
USE_WANDB = True

if USE_WANDB:
    os.environ["WANDB_API_KEY"] = WANDB_TOKEN
    wandb.init(project="legal-rag-slm-finetuning", name=EXPERIMENT_NAME)
    report_target = "wandb"
else:
    os.environ["WANDB_DISABLED"] = "true"
    report_target = "none"

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: kumo11 (kumo11_personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260908_161951-4ucujf0v
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run sft-qwen25-3b-run2
wandb: ⭐️ View project at https://wandb.ai/kumo11_personal/legal-rag-slm-finetuning
wandb: 🚀 View run at https://wandb.ai/kumo11_personal/legal-rag-slm-finetuning/runs/4ucujf0v
wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


# 3. Model & Tokenizer

In [6]:
MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct"
CHAT_TEMPLATE = "qwen-2.5"  # cek unsloth.chat_templates.CHAT_TEMPLATES kalau ganti model family

model, tokenizer = load_model_and_tokenizer(
    model_name=MODEL_NAME,
    chat_template=CHAT_TEMPLATE,
    max_seq_length=2048,
    load_in_4bit=True,
)

==((====))==  Unsloth 2026.9.3: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
total weights      : 2.300 GiB
no_split classes   : ['Qwen2DecoderLayer']
output head        : lm_head -> cuda:1
head headroom      : 0.359 GiB
activation reserve : 11.658 GiB requested
tied to head       : ['model.embed_tokens']
  cuda:0  budget  12.98 GiB  weights  1.266 GiB  free 11.713 GiB  reserve 11.658 GiB
  cuda:1  budget  13.00 GiB  weights  1.034 GiB  free 11.962 GiB  reserve 11.488 GiB   <- output head
note: Bnb4BitHfQuantizer.adjust_max_memory lowered the budget on cuda:0 14.421 -> 12.979 GiB, cuda:1 14.440 -> 12.996 GiB (memory th

In [7]:
# Verifikasi double quantization aktif (requirement brief: "double quantization, 4-bit")
print(model.config.quantization_config)

{'bnb_4bit_compute_dtype': torch.float16, 'bnb_4bit_quant_type': 'nf4', 'bnb_4bit_use_double_quant': True, 'llm_int8_enable_fp32_cpu_offload': False, 'llm_int8_has_fp16_weight': False, 'llm_int8_skip_modules': ['lm_head', 'multi_modal_projector', 'merger', 'modality_projection', 'model.layers.2.mlp', 'model.layers.3.mlp', 'model.layers.30.mlp'], 'llm_int8_threshold': 6.0, 'load_in_4bit': True, 'load_in_8bit': False, 'quant_method': 'bitsandbytes'}


# 4. PEFT (LoRA) Configuration

In [8]:
model = apply_lora(
    model,
    r=8,
    lora_alpha=8,
    lora_dropout=0.0,
)

Unsloth 2026.9.3 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


# 5. Dataset Preparation

Dataset `Ichsan2895/alpaca-gpt4-indonesian` cuma punya kolom `['Unnamed: 0', 'input', 'output']`

In [9]:
system_prompt = "Kamu adalah asisten AI yang menjawab pertanyaan pengguna dengan akurat dan jelas dalam Bahasa Indonesia."

train_dataset, val_dataset = prepare_datasets(
    tokenizer,
    system_prompt=system_prompt,
    test_size=0.05,
    seed=1010,
)

print(f"Jumlah data training  : {len(train_dataset)}")
print(f"Jumlah data validation: {len(val_dataset)}")

README.md: 0.00B [00:00, ?B/s]

alpaca-gpt4-indonesia.csv:   0%|          | 0.00/41.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49969 [00:00<?, ? examples/s]

Map:   0%|          | 0/47470 [00:00<?, ? examples/s]

Map:   0%|          | 0/2499 [00:00<?, ? examples/s]

Jumlah data training  : 47470
Jumlah data validation: 2499


In [10]:
# Sanity Check
print(train_dataset[0]["text"])

<|im_start|>system
Kamu adalah asisten AI yang menjawab pertanyaan pengguna dengan akurat dan jelas dalam Bahasa Indonesia.<|im_end|>
<|im_start|>user
Desain antarmuka pengguna untuk aplikasi manajemen tugas berbasis AI.<|im_end|>
<|im_start|>assistant
Berikut adalah desain dan penjelasan untuk antarmuka pengguna dari aplikasi manajemen tugas berbasis AI:

1. Halaman Utama: Saat membuka aplikasi, pengguna akan disambut dengan halaman utama yang bersih dan terorganisir. Halaman ini menampilkan ringkasan tugas yang masih harus dikerjakan, kategori tugas, dan kotak pencarian. Avatar asisten virtual dapat hadir di pojok layar untuk memberikan bantuan dan menjalankan perintah suara.

2. Pembuatan Tugas: Pengguna dapat dengan cepat menambahkan tugas dengan mengklik tombol "tambah tugas", yang akan membuka formulir di mana mereka dapat mengisi deskripsi tugas, tipe (pekerjaan, pribadi, dll.), tanggal jatuh tempo, tingkat prioritas, dan lainnya. AI dapat memberikan saran berdasarkan kata kunci

# 6. Supervised Fine-Tuning

## 6.1 Training Configuration

* `max_steps=2000` dipilih berdasarkan perhitungan kecepatan training (~3.85 detik/step, diukur dari run percobaan 20 steps) -- lihat catatan keputusan lengkap di README.
* `seed` di-fix eksplisit supaya perbandingan antar eksperimen hyperparameter adil (data
shuffling sama, perbedaan hasil murni dari hyperparameter yang diuji).* `max_steps=2000` dipilih berdasarkan perhitungan kecepatan training (~3.85 detik/step, diukur dari run percobaan 20 steps) -- lihat catatan keputusan lengkap di README.
* `seed` di-fix eksplisit supaya perbandingan antar eksperimen hyperparameter adil (data
shuffling sama, perbedaan hasil murni dari hyperparameter yang diuji).

In [11]:
sft_config = SFTConfig(
    output_dir=f"outputs/{EXPERIMENT_NAME}",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=False,
    warmup_steps=5,
    max_steps=1000,
    learning_rate=2e-4,
    lr_scheduler_type="linear",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=1,
    optim="paged_adamw_8bit",
    dataset_text_field="text",
    max_length=512,
    report_to=report_target,
    run_name=EXPERIMENT_NAME if USE_WANDB else None,

    eval_strategy="steps",
    eval_steps=100,
    per_device_eval_batch_size=2,

    save_strategy="steps",
    save_steps=100, # checkpoint berkala
    save_total_limit=2, # biar gak numpuk checkpoint & penuhin disk
)

In [12]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset, 
    args=sft_config,
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/47470 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2499 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [13]:
trainer.model.print_trainable_parameters()

trainable params: 14,966,784 || all params: 3,100,905,472 || trainable%: 0.4827


## 6.2 Training Run

In [14]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 47,470 | Num Epochs = 1 | Total steps = 1,000
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 14,966,784 of 3,100,905,472 (0.48% trained)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
100,1.170200,1.096309
200,1.133500,1.081231
300,1.066600,1.072127
400,0.928200,1.063421
500,0.865200,1.057507
600,1.055300,1.052683
700,0.914900,1.049042
800,1.155800,1.045612
900,1.173200,1.043499
1000,1.068500,1.042561


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


TrainOutput(global_step=1000, training_loss=1.0977005667090416, metrics={'train_runtime': 8449.1904, 'train_samples_per_second': 0.947, 'train_steps_per_second': 0.118, 'total_flos': 3.69440709272617e+16, 'train_loss': 1.0977005667090416, 'epoch': 0.16852749104697703})

In [15]:
if USE_WANDB:
    wandb.finish()

wandb: updating run metadata
wandb: uploading history steps 1009-1010, summary
wandb: 
wandb: Run history:
wandb:               eval/loss █▆▅▄▃▂▂▁▁▁
wandb:            eval/runtime ▄▁▃██▇▇▇▇▇
wandb: eval/samples_per_second ▅█▆▁▁▂▂▂▂▂
wandb:   eval/steps_per_second ▅█▆▁▁▂▂▂▂▂
wandb:             train/epoch ▁▁▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇███
wandb:       train/global_step ▁▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇█████
wandb:         train/grad_norm ▁▅▃▂▂▄▃▃▃▃▃▃▄▃▅▄▅▄▄▄▆▆▃▅▄▅▆▇▄▇▇▇█▇▇▆█▆▄▇
wandb:     train/learning_rate █████▇▇▆▆▆▆▆▆▆▅▅▅▅▅▅▅▅▅▄▄▄▄▄▄▃▃▂▂▂▂▁▁▁▁▁
wandb:              train/loss ▆█▄▆▄▅▃▄▆▆▆▅▇▅▆▆█▅▅▃▄▅▅▇▄▅▆▆▂▄▂▃▃▁▅▄▃▂▅▃
wandb: 
wandb: Run summary:
wandb:               eval/loss 1.04256
wandb:            eval/runtime 473.6108
wandb: eval/samples_per_second 5.276
wandb:   eval/steps_per_second 2.639
wandb:              total_flos 3.69440709272617e+16
wandb:             train/epoch 0.16853
wandb:       train/global_step 1000
wandb:         train/grad_norm 0.43016
wandb:     train/l

# 7. Push Model to HuggingFace Hub

In [16]:
HF_REPO_ID = f"11erlangga/{EXPERIMENT_NAME}"

model.push_to_hub_merged(
    HF_REPO_ID,
    tokenizer,
    save_method="merged_16bit",
    token=HF_TOKEN,
)

print(f"Model pushed to: https://huggingface.co/{HF_REPO_ID}")

config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Unsloth: Merging weights into 16bit:  50%|█████     | 1/2 [01:25<01:25, 85.06s/it]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:10<00:00, 65.28s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/11erlangga/sft-qwen25-3b-run2`
Model pushed to: https://huggingface.co/11erlangga/sft-qwen25-3b-run2


# 8. Sanity Check (Inference Test)

In [17]:
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048, padding_idx=151654)
        (layers): ModuleList(
          (0-1): 2 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Lin

In [18]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Tolong buatkan draft email profesional kepada atasan untuk mengajukan izin cuti 3 hari karena keperluan keluarga."}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to("cuda")


In [19]:
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

_ = model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    streamer=text_streamer,
    max_new_tokens=256,
    temperature=0.3,
    repetition_penalty=1.2,
    do_sample=True,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

Halo [Atasan],

Saya menulis ini untuk meminta izin Anda tentang waktu liburan saya selama tiga hari mendaki gunung bersama anggota keluarga terdekat. Saya sangat berharap dapat bekerja dari rumah atau menggunakan teknologi lainnya untuk tetap produktif saat tidak di kantor.

Terima kasih atas kesempatan, dan harapan bisa melanjutkan pekerjaan saya tanpa gangguan pada tanggal tersebut.

Dengan hormat,
[Your Name]<|im_end|>
